In [ ]:
import os
import json
from pathlib import Path
from IPython.display import Audio
from typing import Optional, Callable

import matplotlib.pyplot as plt
from huggingface_hub import login as hf_login

import torch
import librosa
from torch.utils.data import Dataset


if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

ROOT_DIR = Path.home() / "workspace"
HF_TOKEN = os.getenv("HF_TOKEN")

hf_login(token=HF_TOKEN)

In [ ]:
ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data" / "clotho-moment"

In [ ]:
# Use train-dataset for exploratory data analysis (EDA)
dataset_sample = DATA_DIR / "train" / "train-000.tar"
dataset_sample.is_file()

In [ ]:
# with tarfile.open(dataset_sample, "r") as tar:
#     tar.extractall(path=DATA_DIR / "train-000")

"""
Note:
Within the file, there are files with the following structure:
- Amsterdam_00_600.json
- Amsterdam_00_600.wav
- Amsterdam_00_610.json
- Amsterdam_00_610.wav
- ...

Overall, considering the code is Amsterdam_00_600, there are .wav and .json file related to it.
"""

In [ ]:
sample_code = "Amsterdam_00_600"
json_path = DATA_DIR / "train-000" / f"{sample_code}.json"
wav_path = DATA_DIR / "train-000" / f"{sample_code}.wav"

In [ ]:
json_data = json.load(open(json_path, "r"))
json_data.keys()

In [ ]:
json_data

In [ ]:
DEFAULT_SAMPLING_RATE = 44_000

y, sr = librosa.load(wav_path, sr=DEFAULT_SAMPLING_RATE)

Audio(y, rate=DEFAULT_SAMPLING_RATE)

In [ ]:
# 2. Time-Domain Analysis (Waveforms)
start_time = 10.222201391753154  # From foreground event
duration = 24.0  # Event duration
end_time = start_time + duration

plt.figure(figsize=(15, 8))
librosa.display.waveshow(y, sr=sr, alpha=0.5, label="Clean")

# Add vertical lines to mark foreground event boundaries
plt.axvline(
    x=start_time,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Event start: {start_time:.2f}s",
)
plt.axvline(
    x=end_time,
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Event end: {end_time:.2f}s",
)

# Shade the region where the event occurs
plt.axvspan(start_time, end_time, alpha=0.2, color="yellow", label="Event duration")

plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## Torch Dataset Implementation

In [ ]:
class ClothoMomentDataset(Dataset):
    def __init__(
        self,
        audio_dir: Path,
        label_dir: Path,
        sample_rate: int = 44100,
        *,
        transform: Optional[Callable] = None,
    ):
        self.audio_dir = audio_dir
        self.label_dir = label_dir
        self.sample_rate = sample_rate
        self.transform = transform

        self.audio_files = sorted(list(self.audio_dir.glob("*.wav")))
        self.label_files = sorted(list(self.label_dir.glob("*.json")))

        self.pair_files = [
            (audio, label)
            for audio in self.audio_files
            for label in self.label_files
            if audio.stem == label.stem.replace("_std", "")
        ]

    def __len__(self):
        return len(self.pair_files)

    def __getitem__(self, idx):
        audio_path, label_path = self.pair_files[idx]

        # Load audio
        waveform, sample_rate = librosa.load(audio_path, sr=self.sample_rate)

        # Load labels
        with open(label_path, "r") as f:
            labels = json.load(f)

        if self.transform:
            waveform = self.transform(waveform)

        return {
            "waveform": waveform,
            "sample_rate": sample_rate,
            "labels": labels,
        }

In [ ]:
sample = dataset[0]
print(f"Sample keys: {sample.keys()}")
print(f"Waveform shape: {sample['waveform'].shape}")
print(f"Sample rate: {sample['sample_rate']}")
print(f"Labels: {sample['labels']}")